
## DinoV3 + PASTIS + Supervisado Multiclase
Autor: Luis Fonseca A01208933

### Paso 1 - Setup

Prepara el terreno: importa librerías, define hiperparámetros y establece la estructura mínima para trabajar con PASTIS. Deja listo el “mapa” de datos y la reproducibilidad.

- **Modelo base:** `facebook/dinov3-vitl16-pretrain-sat493m` (DINOv3 preentrenado en satélites).  
- **Fine-tuning:** head con **LR=1e-4**, backbone con **LR=5e-6**; más adelante se **descongelan las últimas k=3 capas**.  
- **Entrada:** imágenes **256×256** (múltiplo de 16 para ViT-L/16), canales **RGB Sentinel-2** en orden **B04, B03, B02** (`CH_IDX=(2,1,0)`).  
- **Normalización:** estadísticas desde `pastis_qc/norm_stats.json`.  
- **Clases:** IDs **0..19**; se **excluyen 0 y 19** (fondo/vacío). Válidas **1..18** con **`IGNORE=255`**. `ID2NAME` ayuda a reportar IoU por clase.  
- **Rutas:** verifica existencia de `DATA_S2` y `ANNOTATIONS`. Crea `pastis_qc/` para salidas (splits, checkpoints).  
- **Descubrimiento de pares:** usa `S2_<id>.npy` ↔ `ParcelIDs_<id>.npy` y conserva **solo IDs comunes**.  
- **Splits reproducibles:** semilla **1337**, **70% train / 15% val / 15% test**; guarda `pastis_qc/splits.csv` con columnas `id,split,img_relpath,msk_relpath`.  
- **Reproducibilidad global:** fija semillas (Python/NumPy/PyTorch) y activa **cuDNN determinista**.

In [1]:
from pathlib import Path
import re, csv, os, random
import numpy as np
import json, csv, numpy as np
import torch
import csv, functools
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import math
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from torch.optim.lr_scheduler import CosineAnnealingLR
import pandas as pd
import matplotlib.pyplot as plt

# --- hiperparámetros base usados luego (modelo/entrenamiento) ---
MODEL_ID         = "facebook/dinov3-vitl16-pretrain-sat493m"
UNFREEZE_LAST_K = 3    # puedes subir a 3 más adelante
LR_HEAD          = 1e-4
LR_BACKBONE     = 5e-6 
WEIGHT_DECAY     = 1e-4

# Tamaño de entrada (múltiplo de 16 para ViT-L/16)
INPUT_SIZE = 256
# Canales a usar (RGB de Sentinel-2 en tu orden): B04, B03, B02
CH_IDX = (2, 1, 0)
# Archivo con estadísticas calculadas en pasos previos
STATS_JSON = Path("pastis_qc/norm_stats.json")

# PASTIS usa IDs 0..19 (nosotros ignoramos 0 y 19)
N_CLASSES       = 20
EXCLUDE_LABELS  = {0, 19}
IGNORE          = 255
VALID_CLASSES   = [c for c in range(N_CLASSES) if c not in EXCLUDE_LABELS]
ALLOWED_CLASSES = set(range(1, 19))  # 1..18

# (nombres por clase si luego imprimes IoU por clase
ID2NAME = {
    0:"Background", 1:"Meadow", 2:"Soft winter wheat", 3:"Corn", 4:"Winter barley",
    5:"Winter rapeseed", 6:"Spring barley", 7:"Sunflower", 8:"Grapevine", 9:"Beet",
    10:"Winter triticale", 11:"Winter durum wheat", 12:"Fruits, vegetables, flowers",
    13:"Potatoes", 14:"Leguminous fodder", 15:"Soybeans", 16:"Orchard",
    17:"Mixed cereal", 18:"Sorghum", 19:"Void label"
}
print(f"[labels] N_CLASSES={N_CLASSES} | IGNORE={IGNORE} | excluyo={EXCLUDE_LABELS}")

# --- rutas base ---
DATA_ROOT  = Path(r"../PASTIS/PASTIS")
OUT_DIR    = Path("pastis_qc")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_CSV = OUT_DIR / "splits.csv"

# --- validar estructura mínima ---
S2_DIR   = DATA_ROOT / "DATA_S2"
ANN_DIR  = DATA_ROOT / "ANNOTATIONS"
assert S2_DIR.exists(),  f"No existe {S2_DIR}"
assert ANN_DIR.exists(), f"No existe {ANN_DIR}"

# --- utilidades ---
def _extract_id(fname: str, prefix: str) -> int | None:
    # coincide: <prefix>_<ID>.npy
    m = re.match(rf"^{re.escape(prefix)}_(\d+)\.npy$", fname)
    return int(m.group(1)) if m else None

def discover_common_ids(s2_dir: Path, ann_dir: Path):
    img_ids = set()
    msk_ids = set()
    for name in os.listdir(s2_dir):
        if name.endswith(".npy"):
            pid = _extract_id(name, "S2")
            if pid is not None:
                img_ids.add(pid)
    for name in os.listdir(ann_dir):
        if name.endswith(".npy"):
            pid = _extract_id(name, "ParcelIDs")
            if pid is not None:
                msk_ids.add(pid)
    common = sorted(img_ids & msk_ids)
    return common

# --- descubrir IDs y crear splits ---
IDS = discover_common_ids(S2_DIR, ANN_DIR)
n = len(IDS)
assert n > 0, "No se encontraron pares S2_* / ParcelIDs_*."

# split reproducible 70/15/15
SEED = 1337
rng = np.random.RandomState(SEED)
ids = np.array(IDS, dtype=int)
rng.shuffle(ids)

n_train = int(0.70 * n)
n_rest  = n - n_train
n_val   = n_rest // 2
n_test  = n - n_train - n_val

train_ids = ids[:n_train]
val_ids   = ids[n_train:n_train+n_val]
test_ids  = ids[n_train+n_val:]

print(f"[B0] total={n} | train={len(train_ids)} val={len(val_ids)} test={len(test_ids)}")

# escribir splits.csv: id,split,img_relpath,msk_relpath
with open(SPLITS_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id","split","img_relpath","msk_relpath"])
    def row(pid, split):
        img_rel = f"DATA_S2/S2_{pid}.npy"
        msk_rel = f"ANNOTATIONS/ParcelIDs_{pid}.npy"
        return [str(pid), split, img_rel, msk_rel]
    for pid in train_ids: w.writerow(row(pid, "train"))
    for pid in val_ids:   w.writerow(row(pid, "val"))
    for pid in test_ids:  w.writerow(row(pid, "test"))

print(f"[B0] Escribí {SPLITS_CSV} con {n} filas.")
# muestra 3 filas
with open(SPLITS_CSV, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i < 4: print(line.strip())
        else: break

# --- semillas globales (reproducibilidad) ---
def fix_seeds(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
fix_seeds()


[labels] N_CLASSES=20 | IGNORE=255 | excluyo={0, 19}
[B0] total=2433 | train=1703 val=365 test=365
[B0] Escribí pastis_qc\splits.csv con 2433 filas.
id,split,img_relpath,msk_relpath
10346,train,DATA_S2/S2_10346.npy,ANNOTATIONS/ParcelIDs_10346.npy
10138,train,DATA_S2/S2_10138.npy,ANNOTATIONS/ParcelIDs_10138.npy
30562,train,DATA_S2/S2_30562.npy,ANNOTATIONS/ParcelIDs_30562.npy


### Paso 2 — Estadísticas de normalización (rápido y robusto)

Calcula (o carga) la **media** y **desviación estándar** por canal para normalizar las imágenes. Usa la **mediana temporal** para comprimir la serie (reduce nubes/ruido) y acelera tomando una **muestra** del split de entrenamiento.

  - Si existe `pastis_qc/norm_stats.json`, lo **lee** y (si aplica) **subselecciona** los canales definidos en `CH_IDX`.
  - Si no existe, recorre hasta `sample_max=200` entradas del **split train** (`splits.csv`), carga `S2_<id>.npy` con forma `(T,C,H,W)`, aplica **mediana en T** → `(H,W,C)` y se queda con los canales `CH_IDX`.
  - Concatena píxeles y calcula `mean`/`std` con `nan`-safe; añade un **ε=1e-6** a `std` para evitar divisiones por cero.
  - Guarda `{"channels_used", "mean", "std"}` en `norm_stats.json` para reutilizar después.

  - La **mediana temporal** es resistente a outliers y condiciones cambiantes; mejora una normalización **estable**.
  - Tomar una **muestra** acelera sin perder representatividad del entrenamiento.

- **Salida esperada**
  - Archivo `pastis_qc/norm_stats.json` con `mean` y `std` por canal.
  - Mensaje `[stats] Guardé ...` la primera vez; en corridas siguientes solo leerá el JSON.


In [2]:
def _temporal_reduce_median_for_stats(X):  # (T,C,H,W) -> (H,W,C)
    Xc = np.median(X, axis=0)
    return np.transpose(Xc, (1,2,0))

def load_or_compute_stats(stats_json: Path, data_root: Path, splits_csv: Path, ch_idx=CH_IDX, sample_max=200):
    """
    Si existe pastis_qc/norm_stats.json lo lee; si no, calcula
    MEAN/STD en hasta 'sample_max' parches del split train (mediana temporal).
    """
    if stats_json.exists():
        with open(stats_json, "r", encoding="utf-8") as f:
            js = json.load(f)
        mean = np.array([js["mean"][i] for i in range(len(js["mean"]))], dtype=np.float32)
        std  = np.array([js["std"][i]  for i in range(len(js["std"]))],  dtype=np.float32)
        # Si el JSON trae todas las bandas, sub-selecciona CH_IDX
        if mean.shape[0] >= max(ch_idx)+1:
            mean = mean[list(ch_idx)]
            std  = std[list(ch_idx)]
        return mean, std

    # --- calcular si no existe ---
    ids = []
    with open(splits_csv, "r", encoding="utf-8") as f:
        rdr = csv.DictReader(f)
        for r in rdr:
            if r["split"] == "train":
                ids.append(r)
            if len(ids) >= sample_max:
                break

    acc = []
    for r in ids:
        X = np.load(data_root / r["img_relpath"])     # (T,C,H,W)
        x_hw_c = _temporal_reduce_median_for_stats(X) # (H,W,C)
        x_sel  = x_hw_c[:, :, list(ch_idx)].astype(np.float32)
        acc.append(x_sel.reshape(-1, x_sel.shape[-1]))

    arr  = np.concatenate(acc, axis=0)
    mean = np.nanmean(arr, axis=0).astype(np.float32)
    std  = (np.nanstd(arr, axis=0) + 1e-6).astype(np.float32)

    stats_json.parent.mkdir(parents=True, exist_ok=True)
    with open(stats_json, "w", encoding="utf-8") as f:
        json.dump({"channels_used": list(ch_idx),
                   "mean": mean.tolist(),
                   "std": std.tolist()}, f, ensure_ascii=False, indent=2)
    print(f"[stats] Guardé {stats_json} con mean/std de {len(ids)} parches.")
    return mean, std

# Cargar (o calcular) estadísticas
MEAN, STD = load_or_compute_stats(STATS_JSON, DATA_ROOT, SPLITS_CSV, ch_idx=CH_IDX)
print(f"[conf] INPUT_SIZE={INPUT_SIZE} | CH_IDX={CH_IDX} | MEAN={MEAN.round(4)} | STD={STD.round(4)}")

# DEVICE para cuando armes el modelo
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[conf] DEVICE={DEVICE}")


[stats] Guardé pastis_qc\norm_stats.json con mean/std de 200 parches.
[conf] INPUT_SIZE=256 | CH_IDX=(2, 1, 0) | MEAN=[786.3094 741.4712 474.2075] | STD=[407.845  270.9659 218.4496]
[conf] DEVICE=cuda


### Paso 3 — Auditoría de etiquetas (TARGET) y estimación de K

Revisa cómo vienen las **máscaras de clase** en disco y estima cuántas **clases** aparenta tener el dataset (K ≈ `max_label + 1`). Útil para detectar **nombres de archivo** distintos, **formas raras** y **canales extra** en las máscaras.

  - Busca el archivo de máscara por ID aceptando **variantes de nombre** (`TARGET_*`, `Labels_*`, mayúsc./minúsc.).
  - Armoniza cualquier forma de máscara a **(C,H,W)**.
  - Elige el **canal de clases** con una heurística simple (prioriza valores enteros en rangos cortos, 0..19/25).
  - Muestra un **resumen por patch** (min/max/únicos) y calcula `K = max_label + 1`.

  - Valida que las máscaras estén **presentes**, con **canal correcto** y sin rangos inesperados.
  - Avisa si hay parches **sin TARGET** o con **formatos atípicos**.
  - Permite ajustar la **cabeza del modelo** si el número de clases real difiere de lo esperado.

  - Lee IDs desde `splits.csv` (muestra hasta `sample_max`).
  - Para cada ID: carga, normaliza a (C,H,W), **detecta canal** de clase; si no lo logra, usa **canal 0** como respaldo.
  - Acumula el **máximo label** observado y reporta `K_est`.

> Nota: En PASTIS se espera que los labels estén en **0..19**. Esta auditoría ayuda a confirmar esa expectativa antes de definir la **head de segmentación**.


In [3]:
def _find_target_path(root: Path, pid: str):
    """Localiza TARGET_<ID>.npy o variantes aceptadas."""
    cands = [
        root / f"ANNOTATIONS/TARGET_{pid}.npy",
        root / f"ANNOTATIONS/target_{pid}.npy",
        root / f"annotations/TARGET_{pid}.npy",
        root / f"annotations/target_{pid}.npy",
        # fallbacks
        root / f"ANNOTATIONS/Labels_{pid}.npy",
        root / f"ANNOTATIONS/labels_{pid}.npy",
        root / f"annotations/Labels_{pid}.npy",
        root / f"annotations/labels_{pid}.npy",
    ]
    for p in cands:
        if p.exists():
            return p
    return None

def _to_chw(y: np.ndarray) -> np.ndarray:
    """Convierte cualquier (H,W)/(H,W,C)/(C,H,W) → (C,H,W)."""
    y = np.asarray(y)
    if y.ndim == 2:
        return y[None, ...]
    if y.ndim == 3:
        if y.shape[0] <= 64 and y.shape[1] >= 8 and y.shape[2] >= 8:
            return y                          # (C,H,W)
        if y.shape[-1] <= 64:
            return np.transpose(y, (2,0,1))   # (H,W,C)→(C,H,W)
        return np.moveaxis(y, np.argmin(y.shape), 0)
    raise ValueError(f"TARGET ndim={y.ndim} no soportado")

def _pick_class_channel(chw: np.ndarray, hard_k_max=25, need_frac=0.98):
    """
    Elige el canal 'clase' con esta lógica:
      1) Fuerte: ≥98% de pixeles en [0..25] y ≤25 clases únicas.
         (si varios, más únicos y menor máximo)
      2) Suave: el de mayor fracción en [0..19] (si ≥0.50), desempata por menor máx y menos únicos.
      3) Último: canal con menor máx y pocos únicos si ambos ≤hard_k_max.
      4) Si nada cuadra, devuelve None.
    """
    C,H,W = chw.shape
    in25 = np.arange(hard_k_max + 1)
    in20 = np.arange(20)

    strong = []
    stats = []
    for ci in range(C):
        ch = chw[ci]
        uniq = np.unique(ch); nuniq = uniq.size
        frac25 = np.isin(ch, in25).mean()
        frac20 = np.isin(ch, in20).mean()
        mx   = int(ch.max())
        stats.append((ci, nuniq, mx, frac25, frac20))
        if frac25 >= need_frac and nuniq <= hard_k_max:
            score = (nuniq, -mx, frac25)  # más únicos (hasta 25), menor máx, mayor fracción
            strong.append((score, ci))

    if strong:
        strong.sort(reverse=True)
        return strong[0][1]

    # fallback suave: maximiza cobertura en 0..19
    stats.sort(key=lambda t: (t[4], -t[2], -t[1]), reverse=True)  # frac20 desc, mx asc, nuniq asc
    best_ci, nuniq, mx, frac25, frac20 = stats[0]
    if frac20 >= 0.50:
        return best_ci

    # último recurso: menor máximo y pocos únicos
    stats2 = sorted(stats, key=lambda t: (t[2], t[1]))  # mx asc, nuniq asc
    best_ci2, nuniq2, mx2, _, _ = stats2[0]
    if mx2 <= hard_k_max and nuniq2 <= hard_k_max:
        return best_ci2

    return None

def audit_targets_and_estimate_k(data_root: Path, splits_csv: Path, sample_max=30):
    """
    Audita algunos TARGETs y estima K = (max_label + 1) del canal detectado.
    Imprime resumen por patch y devuelve K_est (o None si no encontró).
    """
    # IDs de muestra desde splits.csv
    ids = []
    with open(splits_csv, "r", encoding="utf-8") as f:
        rdr = csv.DictReader(f)
        for r in rdr:
            ids.append(str(r["id"]))
    ids = ids[:max(1, sample_max)]

    found, max_label = 0, -1
    print(f"[B1] Auditando hasta {len(ids)} patches...")
    for pid in ids:
        p = _find_target_path(data_root, pid)
        if p is None:
            print(f"[B1][{pid}] ❌ TARGET no encontrado")
            continue
        y_raw = np.load(p)
        chw   = _to_chw(y_raw)
        c_cls = _pick_class_channel(chw, hard_k_max=25, need_frac=0.98)
        if c_cls is None:
            # Si no detecta, canal 0 por defecto (comportamiento que usamos como fallback)
            print(f"[B1][{pid}] ⚠️  {p.name} | raw{y_raw.shape} | NO se detectó canal de clase → usar canal 0 (fallback)")
            y_hw = chw[0].astype(np.int64)
        else:
            y_hw = chw[c_cls].astype(np.int64)

        uniq = np.unique(y_hw)
        found += 1
        max_label = max(max_label, int(y_hw.max()))
        show = uniq[:20] if uniq.size > 20 else uniq
        print(f"[B1][{pid}] ✅ {p.name} | raw{y_raw.shape} -> hw{y_hw.shape} | "
              f"canal_clase={c_cls if c_cls is not None else 0} | min={int(y_hw.min())} max={int(y_hw.max())} | uniq({len(uniq)}): {show}")

    if found == 0:
        print("[B1] ❗ No se detectó ningún TARGET utilizable en la muestra.")
        return None
    K = int(max_label) + 1
    print(f"[B1] ► Estimación N_CLASSES logits = {K} (a partir de max_label={max_label})")
    return K

# --- Ejecuta auditoría ---
K_est = audit_targets_and_estimate_k(DATA_ROOT, SPLITS_CSV, sample_max=30)



[B1] Auditando hasta 30 patches...
[B1][10346] ✅ TARGET_10346.npy | raw(3, 128, 128) -> hw(128, 128) | canal_clase=0 | min=0 max=19 | uniq(6): [ 0  1  3 16 17 19]
[B1][10138] ✅ TARGET_10138.npy | raw(3, 128, 128) -> hw(128, 128) | canal_clase=0 | min=0 max=19 | uniq(6): [ 0  1  3  6 14 19]
[B1][30562] ✅ TARGET_30562.npy | raw(3, 128, 128) -> hw(128, 128) | canal_clase=0 | min=0 max=19 | uniq(6): [ 0  1  3  6 15 19]
[B1][10327] ✅ TARGET_10327.npy | raw(3, 128, 128) -> hw(128, 128) | canal_clase=0 | min=0 max=19 | uniq(7): [ 0  1  2  3  4 11 19]
[B1][30519] ✅ TARGET_30519.npy | raw(3, 128, 128) -> hw(128, 128) | canal_clase=0 | min=0 max=19 | uniq(8): [ 0  1  3  5 14 15 17 19]
[B1][30629] ✅ TARGET_30629.npy | raw(3, 128, 128) -> hw(128, 128) | canal_clase=0 | min=0 max=19 | uniq(4): [ 0  1  4 19]
[B1][30464] ✅ TARGET_30464.npy | raw(3, 128, 128) -> hw(128, 128) | canal_clase=0 | min=0 max=19 | uniq(8): [ 0  1  2  3  4  5 15 19]
[B1][40544] ✅ TARGET_40544.npy | raw(3, 128, 128) -> hw(128,

### Paso 4 — Dataset y DataLoaders (PASTIS desde `TARGET_*`)

Construye un **Dataset** que toma imágenes y máscaras de PASTIS directamente desde `S2_<id>.npy` y `TARGET_<id>.npy` (o variantes), y entrega tensores listos para el modelo.

**Flujo general**
- **Imagen** `(T,C,H,W)` → **mediana temporal** → `(H,W,C)` → **selección de canales** `CH_IDX` → **normalización** con `MEAN/STD` → **resize bilinear** a `INPUT_SIZE` → tensor `(Csel, S, S)`.
- **Máscara**: localiza `TARGET/Labels`, convierte a `(C,H,W)`, **detecta y cachea** el canal de **clase**, toma `(H,W)` → **remapea** todo lo que no sea `1..18` a **`IGNORE=255`** (excluye explícitamente `0` y `19`) → **resize nearest** a `INPUT_SIZE` → tensor `(S, S)`.

**Puntos clave**
- **Mediana temporal** reduce nubes/ruido sin depender de un instante fijo.
- **Detección de canal de clase** por heurística y **cache** (`lru_cache`): evita recalcular por ID.
- **Normalización** usa `MEAN/STD` calculados antes y respeta `CH_IDX=(2,1,0)` (B04, B03, B02).
- **Tipos y resize seguros**: máscaras pasan a `uint8` para PIL y se reescalan con **nearest** (no mezcla clases); imágenes con **bilinear** (suave para valores continuos).
- **Remapeo de etiquetas**: cualquier valor fuera de `{1..18}` → `IGNORE=255`. Se excluyen `0` y `19`.

**Salida del `__getitem__`**
- `image`: `FloatTensor (Csel, S, S)`
- `mask`: `LongTensor (S, S)` con `255=IGNORE`
- `id`: string del parche

**Sanity-check y loaders**
- Imprime tamaños y **clases válidas** detectadas en 2–3 muestras del train.
- Crea `DataLoader` para `train/val` (batch size 4, `shuffle=True` en train).

In [4]:

try:
    RESAMPLE_BILINEAR = Image.Resampling.BILINEAR
    RESAMPLE_NEAREST  = Image.Resampling.NEAREST
except Exception:
    RESAMPLE_BILINEAR = Image.BILINEAR
    RESAMPLE_NEAREST  = Image.NEAREST

# --- helpers de imagen ---
def _temporal_reduce_median(X):   # (T,C,H,W) -> (H,W,C)
    Xc = np.median(X, axis=0)
    return np.transpose(Xc, (1,2,0))

def _norm_select(x_hw_c):
    # Usa canales definidos en CH_IDX y normaliza con MEAN/STD
    x = x_hw_c[:, :, list(CH_IDX)].astype(np.float32)
    return (x - MEAN) / (STD + 1e-6)

# --- detección por id ---
@functools.lru_cache(maxsize=4096)
def detect_class_channel_for_id(pid: str) -> int:
    """Detecta y cachea el índice de canal de CLASE para un patch id dado."""
    p = _find_target_path(DATA_ROOT, pid)
    if p is None:
        raise FileNotFoundError(f"No hallé TARGET/Labels para id={pid}")
    y = np.load(p)
    chw = _to_chw(y)
    c = _pick_class_channel(chw, hard_k_max=25, need_frac=0.98)
    if c is None:
        print(f"[warn] No pude detectar canal de clase para id={pid}; uso canal 0 por defecto.")
        c = 0
    return int(c)

def load_class_mask_from_target(root: Path, pid: str) -> np.ndarray:
    """Carga TARGET y devuelve máscara de CLASE (H,W) como int64."""
    p = _find_target_path(root, pid)
    if p is None:
        raise FileNotFoundError(f"No hallé TARGET/Labels para id={pid}")
    y = np.load(p)
    chw = _to_chw(y)
    c_cls = detect_class_channel_for_id(pid)
    y_cls = chw[c_cls].astype(np.int64)
    return y_cls

# --- Dataset ---
class PastisMulticlassDatasetTarget(Dataset):
    def __init__(self, data_root: Path, splits_csv: Path, split="train", input_size=256):
        self.root = Path(data_root)
        self.S = int(input_size)
        rows = []
        with open(splits_csv, "r", encoding="utf-8") as f:
            rdr = csv.DictReader(f)
            for r in rdr:
                if r["split"] == split:
                    rows.append(r)
        assert rows, f"split vacío: {split}"
        self.rows = rows

    def __len__(self): return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        pid = str(r["id"])

        # Imagen (T,C,H,W) -> (H,W,C) median temporal
        X = np.load(self.root / r["img_relpath"])
        x_hw_c = _temporal_reduce_median(X)

        # Clase (H,W) desde TARGET canal-CLASE
        y_cls = load_class_mask_from_target(self.root, pid).astype(np.int64)

        # 1) Remapear TODO lo que no sea 1..18 a IGNORE (incluye >19)
        mask_invalid = ~np.isin(y_cls, list(ALLOWED_CLASSES) + list(EXCLUDE_LABELS))
        y_cls[mask_invalid] = IGNORE

        # 2) Excluir explícitamente 0 y 19 -> IGNORE
        for lbl in EXCLUDE_LABELS:
            y_cls[y_cls == lbl] = IGNORE

        # --- PIL no acepta int64: usar uint8 para resize categórico ---
        y_u8 = y_cls.astype(np.uint8)

        # Normalización + resize
        x = _norm_select(x_hw_c)                 # (H,W,Csel)

        # resize de canales continuos (bilinear)
        xs = []
        for c in range(x.shape[-1]):
            im = Image.fromarray(x[..., c])
            im = im.resize((self.S, self.S), resample=RESAMPLE_BILINEAR)
            xs.append(np.array(im, dtype=np.float32))
        xS = np.stack(xs, axis=0)                # (Csel,S,S)

        # resize de máscara categórica (nearest) en modo 'L'
        ym = Image.fromarray(y_u8, mode='L')
        ym = ym.resize((self.S, self.S), resample=RESAMPLE_NEAREST)
        yS = np.array(ym, dtype=np.int64)        # (S,S) con 255=IGNORE

        return {"image": torch.from_numpy(xS).float(),
                "mask":  torch.from_numpy(yS).long(),
                "id": pid}

# --- Construir loaders + sanity-check ---
# Asegúrate de haber definido antes: INPUT_SIZE, CH_IDX, MEAN, STD
train_ds = PastisMulticlassDatasetTarget(DATA_ROOT, SPLITS_CSV, split="train", input_size=INPUT_SIZE)
val_ds   = PastisMulticlassDatasetTarget(DATA_ROOT, SPLITS_CSV, split="val",   input_size=INPUT_SIZE)

print(f"[B2] train={len(train_ds)} | val={len(val_ds)} | INPUT_SIZE={INPUT_SIZE} | CH_IDX={CH_IDX}")

ncheck = min(len(train_ds), 3)
for idx in range(ncheck):
    s = train_ds[idx]
    Xs, ys, pid = s["image"].numpy(), s["mask"].numpy(), s["id"]
    u = sorted(set(np.unique(ys)) - {IGNORE})
    print(f"[B2][train idx {idx}] id={pid} | X={Xs.shape} float32 | y={ys.shape} int64 | clases(validas)={u[:10]}{'...' if len(u)>10 else ''}")

# (si quieres)
from torch.utils.data import DataLoader
train_dl = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=0)


[B2] train=1703 | val=365 | INPUT_SIZE=256 | CH_IDX=(2, 1, 0)
[B2][train idx 0] id=10346 | X=(3, 256, 256) float32 | y=(256, 256) int64 | clases(validas)=[np.int64(1), np.int64(3), np.int64(16), np.int64(17)]
[B2][train idx 1] id=10138 | X=(3, 256, 256) float32 | y=(256, 256) int64 | clases(validas)=[np.int64(1), np.int64(3), np.int64(6), np.int64(14)]
[B2][train idx 2] id=30562 | X=(3, 256, 256) float32 | y=(256, 256) int64 | clases(validas)=[np.int64(1), np.int64(3), np.int64(6), np.int64(15)]


### Paso 4 — Dataset y DataLoaders (PASTIS desde `TARGET_*`)

Construye un **Dataset** que toma imágenes y máscaras de PASTIS directamente desde `S2_<id>.npy` y `TARGET_<id>.npy` (o variantes), y entrega tensores listos para el modelo.

**Flujo general**
- **Imagen** `(T,C,H,W)` → **mediana temporal** → `(H,W,C)` → **selección de canales** `CH_IDX` → **normalización** con `MEAN/STD` → **resize bilinear** a `INPUT_SIZE` → tensor `(Csel, S, S)`.
- **Máscara**: localiza `TARGET/Labels`, convierte a `(C,H,W)`, **detecta y cachea** el canal de **clase**, toma `(H,W)` → **remapea** todo lo que no sea `1..18` a **`IGNORE=255`** (excluye explícitamente `0` y `19`) → **resize nearest** a `INPUT_SIZE` → tensor `(S, S)`.

**Puntos clave**
- **Mediana temporal** reduce nubes/ruido sin depender de un instante fijo.
- **Detección de canal de clase** por heurística y **cache** (`lru_cache`): evita recalcular por ID.
- **Normalización** usa `MEAN/STD` calculados antes y respeta `CH_IDX=(2,1,0)` (B04, B03, B02).
- **Tipos y resize seguros**: máscaras pasan a `uint8` para PIL y se reescalan con **nearest** (no mezcla clases); imágenes con **bilinear** (suave para valores continuos).
- **Remapeo de etiquetas**: cualquier valor fuera de `{1..18}` → `IGNORE=255`. Se excluyen `0` y `19`.

**Salida del `__getitem__`**
- `image`: `FloatTensor (Csel, S, S)`
- `mask`: `LongTensor (S, S)` con `255=IGNORE`
- `id`: string del parche

**Sanity-check y loaders**
- Imprime tamaños y **clases válidas** detectadas en 2–3 muestras del train.
- Crea `DataLoader` para `train/val` (batch size 4, `shuffle=True` en train).


In [5]:
class InputAdapter(nn.Module):
    """Mapea C_in → C_model con una Conv1x1. Si coinciden, identidad."""
    def __init__(self, c_in: int, c_out: int):
        super().__init__()
        self.c_in = c_in
        self.c_out = c_out
        if c_in == c_out:
            self.adapt = None
        else:
            self.adapt = nn.Conv2d(c_in, c_out, kernel_size=.25, bias=False) # 

    def forward(self, x):
        # x: (B, C_in, H, W)
        if self.adapt is None:
            return x
        return self.adapt(x)

class DinoV3SegMC(nn.Module):
    def __init__(self, backbone: nn.Module, num_classes: int, in_channels: int, unfreeze_last_k=0):
        super().__init__()
        self.backbone = backbone
        cfg  = getattr(self.backbone, "config", None)
        hdim = int(getattr(cfg, "hidden_size", 0))
        self.nreg  = int(getattr(cfg, "num_register_tokens", 0))  # p.ej., 4
        self.patch = int(getattr(cfg, "patch_size", 16))          # p.ej., 16
        self.c_exp = int(getattr(cfg, "num_channels", 3))         # p.ej., 3

        # Adaptador de entrada (Csel → C_backbone esperado)
        self.input_adapter = InputAdapter(in_channels, self.c_exp)

        # Cabeza de segmentación ligera
        self.head  = nn.Sequential(
            nn.Conv2d(hdim, hdim // 2, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(hdim // 2, num_classes, 1),
        )

        # Congelar todo el backbone por defecto
        for p in self.backbone.parameters():
            p.requires_grad = False

        # Descongelar últimos K bloques (detección robusta en ViT)
        if unfreeze_last_k > 0:
            from collections import deque
            def find_blocklists(module: nn.Module, min_len=6):
                outs=[]; q=deque([("",module)])
                while q:
                    base,m=q.popleft()
                    for n,ch in m.named_children():
                        p=f"{base}.{n}" if base else n
                        if isinstance(ch, nn.ModuleList) and len(ch) >= min_len:
                            # heurística: primer bloque tiene attn/mlp/norm
                            first=ch[0]; score=0
                            for sn,_ in first.named_modules():
                                s=sn.lower()
                                if "attn" in s or "attention" in s: score+=1
                                if any(k in s for k in ("mlp","ffn","feedforward")): score+=1
                                if "norm" in s or "layernorm" in s or "ln" in s: score+=1
                            outs.append((p, ch, score))
                        q.append((p,ch))
                outs.sort(key=lambda t: (t[2], len(t[1])), reverse=True)
                return outs

            cands = find_blocklists(self.backbone, min_len=6)
            assert cands, "No hallé lista de bloques tipo ViT en el backbone."
            path, blist, _ = cands[0]
            k = min(unfreeze_last_k, len(blist))
            for bi in range(len(blist)-k, len(blist)):
                for _,p in blist[bi].named_parameters():
                    p.requires_grad = True
            # también permitir grad en normalizaciones de salida (suele ayudar)
            for nm,p in self.backbone.named_parameters():
                nml = nm.lower()
                if any(t in nml for t in ["ln_f", "layernorm", "post_layernorm", "norm.weight", "norm.bias"]):
                    p.requires_grad = True
            print(f"[model] Unfreeze últimos {k} bloques en '{path}'")

    def forward(self, x):   # x: (B,Csel,S,S)
        # 1) Adaptar canales al esperado por el backbone
        x_in = self.input_adapter(x)             # (B,C_exp,S,S)

        # 2) Pasar por ViT; salida: (B, 1 + nreg + Npatches, hdim)
        out  = self.backbone(pixel_values=x_in)
        toks = out.last_hidden_state[:, 1 + self.nreg :, :]  # quitar CLS + register tokens

        # 3) Rehacer grilla de patches con patch_size del backbone
        B, N, C = toks.shape
        S = x.shape[-1]
        Ht = Wt = S // self.patch                     # grilla esperada por el patch embed
        assert Ht * Wt == N, f"Tokens={N} no coincide con grid esperada {Ht}x{Wt} (S={S}, patch={self.patch})"

        feat = toks.transpose(1, 2).reshape(B, C, Ht, Wt)    # (B, hdim, Ht, Wt)

        # 4) Cabeza de segmentación y upsample al tamaño de entrada
        log  = self.head(feat)                               # (B, K, Ht, Wt)
        return F.interpolate(log, size=(S, S), mode="bilinear", align_corners=False)

# ----------------------------------------------------------
# Construcción del modelo + optimizador
#   (usa tus globals: MODEL_ID, N_CLASSES, UNFREEZE_LAST_K,
#    LR_HEAD, LR_BACKBONE, WEIGHT_DECAY, DEVICE, CH_IDX)
# ----------------------------------------------------------
print(f"[model] Cargando backbone: {MODEL_ID}")
backbone = AutoModel.from_pretrained(MODEL_ID)

in_channels = len(CH_IDX)          # 4 (NIR+RGB)
model = DinoV3SegMC(
    backbone=backbone,
    num_classes=N_CLASSES,
    in_channels=in_channels,
    unfreeze_last_k=UNFREEZE_LAST_K
).to(DEVICE)

# Param groups: head (lr alto) + lo que se descongele del backbone (lr bajo)
pg_head = [p for p in model.head.parameters()] + \
          ([p for p in model.input_adapter.parameters()] if model.input_adapter.adapt is not None else [])
pg_bb   = [p for p in model.backbone.parameters() if p.requires_grad]

optim   = torch.optim.AdamW(
    [{"params": pg_head, "lr": LR_HEAD, "weight_decay": WEIGHT_DECAY},
     {"params": pg_bb,   "lr": LR_BACKBONE, "weight_decay": WEIGHT_DECAY}],
)

print(f"[model] listo: hdim={getattr(backbone.config,'hidden_size', 'unk')} | "
      f"patch={getattr(backbone.config,'patch_size','unk')} | "
      f"c_exp={getattr(backbone.config,'num_channels','unk')} | "
      f"in_channels={in_channels} | "
      f"ft_blocks={UNFREEZE_LAST_K} | "
      f"pg_head={sum(p.numel() for p in pg_head)} | pg_bb_trainable={sum(p.numel() for p in pg_bb)}")


[model] Cargando backbone: facebook/dinov3-vitl16-pretrain-sat493m
[model] Unfreeze últimos 3 bloques en 'layer'
[model] listo: hdim=1024 | patch=16 | c_exp=3 | in_channels=3 | ft_blocks=3 | pg_head=4729364 | pg_bb_trainable=37793792


### Paso 5 — Pesos por clase (balanceo de Cross-Entropy)

Calcula **pesos por clase** a partir de lo que ve el `DataLoader` para que la pérdida **no favorezca** a las clases más frecuentes.

- **Qué hace**
  - Recorre hasta `max_batches=200` batches y arma un **histograma** de etiquetas (omite `IGNORE=255`).
  - Evita ceros con `max(.,1)` y calcula **pesos inversos normalizados** para las clases **1..18**.
  - Mantiene peso **1.0** para `0` y `19` (no las usamos para aprender).

- **Por qué sirve**
  - Si algunas clases aparecen poco, el modelo tiende a **ignorarlas**. Con pesos inversos, cada error en clases raras **pesa más** y el aprendizaje se equilibra.

- **Salida**
  - `torch.FloatTensor` de longitud `n_classes` con los pesos por índice de clase.



In [6]:
def compute_class_weights(dl, n_classes=20, ignore_idx=255, max_batches=200):
    hist = np.zeros(n_classes, np.float64)
    seen = 0
    for b in dl:
        y = b["mask"].numpy().ravel()
        y = y[(y != ignore_idx) & (y >= 0) & (y < n_classes)]
        if y.size: hist += np.bincount(y, minlength=n_classes)
        seen += 1
        if seen >= max_batches: break
    hist = np.maximum(hist, 1)
    w = np.ones(n_classes, np.float32)
    valid = range(1,19)
    inv = (1.0 / hist[valid]); inv /= inv.mean()
    for i,c in enumerate(valid): w[c] = inv[i]
    return torch.from_numpy(w)

### Paso 6 — Sanity-check, entrenamiento y evaluación (mIoU sin 0/19)

Activa la **pérdida con pesos de clase**, configura **AMP** (precisión mixta) y ejecuta un ciclo de **entrenamiento + validación** con:
- **Cosine LR** (suaviza el aprendizaje por época).
- **Early stopping** (paciencia 6).
- **Mejor checkpoint** guardado en `pastis_qc/best_mc.pth`.
- **mIoU** calculado **sin** las clases `0` y `19` (se ignoran) y con `IGNORE=255`.

**Flujo**
1. **Pérdida**: `CrossEntropyLoss(weight=class_weights, ignore_index=IGNORE)`.  
   Muestra los **pesos 1..18** para verificar el balanceo.
2. **Sanity-check**: un `forward` sobre un batch de `train_dl` y confirma que los `logits` tengan forma `(B, N_CLASSES, S, S)`.
3. **Evaluación (`eval_loop`)**:
   - Modo `eval`, sin gradientes y con AMP.
   - Calcula **loss** y **IoU por clase** solo para `VALID_CLASSES = 1..18`.
   - **mIoU** = promedio simple de esas IoUs.
4. **Entrenamiento**:
   - Modo `train`, AMP (`autocast`) + `GradScaler`.
   - `zero_grad(set_to_none=True)` → paso de optimización con `scaler`.
   - `scheduler.step()` al final de cada época.
   - Se imprime: `train_loss`, `val_loss`, **mIoU(sin 0/19)** y LRs de los grupos.
   - Si el **mIoU** mejora, se **guarda checkpoint** con estado de `model` y `optim`; si no mejora en `early_patience`, se detiene.

**Salida final**
- Resumen con **IoU por clase** (`1..18`) y el **mejor mIoU** alcanzado.
- Archivo `pastis_qc/best_mc.pth` con `{"model", "optim", "miou", "epoch"}`.


In [7]:
# pérdida, AMP y scaler
weights = compute_class_weights(train_dl, n_classes=N_CLASSES, ignore_idx=IGNORE).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=weights, ignore_index=IGNORE)
print("[loss] class weights(1..18):", weights[1:19].detach().cpu().numpy().round(2))
use_amp   = torch.cuda.is_available()
scaler    = torch.cuda.amp.GradScaler(enabled=use_amp)

# -------------------------
# sanity-check de forward
# -------------------------
model.eval()
with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):
    sb = next(iter(train_dl))
    logits = model(sb["image"].to(DEVICE))  # (B,K,S,S)
print(f"[Sanity] logits {tuple(logits.shape)} (ok si K={N_CLASSES})")

# -------------------------
# evaluación (mIoU sin 0/19)
# -------------------------
@torch.no_grad()
def eval_loop(dl):
    model.eval()
    total_loss = 0.0
    inter = {c: 0 for c in VALID_CLASSES}
    union = {c: 0 for c in VALID_CLASSES}

    for b in dl:
        X = b["image"].to(DEVICE)
        y = b["mask"].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(X)
            loss   = criterion(logits, y)

        total_loss += float(loss.item())
        pred = torch.argmax(logits, dim=1).cpu().numpy()
        gt   = y.cpu().numpy()
        valid = (gt != IGNORE)

        for c in VALID_CLASSES:
            p = (pred == c)
            g = (gt == c)
            inter[c] += np.logical_and(p, np.logical_and(g, valid)).sum()
            union[c] += np.logical_or(p, g)[valid].sum()

    ious = []
    for c in VALID_CLASSES:
        iou_c = float(inter[c]) / float(union[c]) if union[c] > 0 else 0.0
        ious.append((c, iou_c))
    miou = float(np.mean([v for _, v in ious])) if ious else 0.0
    return {"loss": total_loss / max(1,len(dl)),
            "miou": miou,
            "iou_per_class": ious}

# -------------------------
# training loop + checkpoint
# -------------------------

TOTAL_EPOCHS = 1
scheduler = CosineAnnealingLR(optim, T_max=TOTAL_EPOCHS)

early_patience = 6
best_miou = -1.0
best_epoch = 0
BEST_CKPT = "pastis_qc/best_mc.pth"

for ep in range(1, TOTAL_EPOCHS + 1):
    model.train(); run = 0.0
    for b in train_dl:
        X = b["image"].to(DEVICE); y = b["mask"].to(DEVICE)
        optim.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=True):
            logits = model(X); loss = criterion(logits, y)
        scaler.scale(loss).backward()
        scaler.step(optim); scaler.update()
        run += float(loss.item())

    scheduler.step()

    val = eval_loop(val_dl)
    tr = run / max(1, len(train_dl))
    print(f"[MC][Ep {ep}/{TOTAL_EPOCHS}] lr={[g['lr'] for g in optim.param_groups]} | "
          f"train_loss={tr:.4f} | val_loss={val['loss']:.4f} | mIoU(sin 0/19)={val['miou']:.3f}")

    if val["miou"] > best_miou + 1e-4:
        best_miou = val["miou"]; best_epoch = ep
        torch.save({"model": model.state_dict(),
                    "optim": optim.state_dict(),
                    "miou": best_miou,
                    "epoch": ep}, BEST_CKPT)
        print(f"[MC] 🔥 Nuevo best mIoU={best_miou:.3f} → {BEST_CKPT}")
    elif ep - best_epoch >= early_patience:
        print(f"[MC] Early stopping. Mejor mIoU={best_miou:.3f} (ep {best_epoch}).")
        break

val = eval_loop(val_dl)
print("\nIoU por clase (sin 0 y 19):")
for cid, iou in val["iou_per_class"]:
    print(f" {cid:>2} {ID2NAME.get(cid, str(cid)):30s} IoU={iou:.3f}")


[loss] class weights(1..18): [0.06 0.17 0.1  0.42 0.66 1.41 0.98 0.32 1.55 1.15 0.66 0.65 2.62 0.67
 0.99 0.84 2.   2.73]


C:\Users\LuisA\AppData\Local\Temp\ipykernel_10320\671936360.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler    = torch.cuda.amp.GradScaler(enabled=use_amp)
C:\Users\LuisA\AppData\Local\Temp\ipykernel_10320\671936360.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=use_amp):


[Sanity] logits (4, 20, 256, 256) (ok si K=20)


C:\Users\LuisA\AppData\Local\Temp\ipykernel_10320\671936360.py:72: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True):
C:\Users\LuisA\AppData\Local\Temp\ipykernel_10320\671936360.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


[MC][Ep 1/1] lr=[0.0, 0.0] | train_loss=2.5147 | val_loss=2.3631 | mIoU(sin 0/19)=0.126
[MC] 🔥 Nuevo best mIoU=0.126 → pastis_qc/best_mc.pth

IoU por clase (sin 0 y 19):
  1 Meadow                         IoU=0.448
  2 Soft winter wheat              IoU=0.143
  3 Corn                           IoU=0.022
  4 Winter barley                  IoU=0.076
  5 Winter rapeseed                IoU=0.135
  6 Spring barley                  IoU=0.019
  7 Sunflower                      IoU=0.133
  8 Grapevine                      IoU=0.381
  9 Beet                           IoU=0.084
 10 Winter triticale               IoU=0.078
 11 Winter durum wheat             IoU=0.125
 12 Fruits, vegetables, flowers    IoU=0.111
 13 Potatoes                       IoU=0.057
 14 Leguminous fodder              IoU=0.087
 15 Soybeans                       IoU=0.107
 16 Orchard                        IoU=0.184
 17 Mixed cereal                   IoU=0.033
 18 Sorghum                        IoU=0.051


### Paso 7 — Visualización en validación y export de métricas

Carga el **mejor checkpoint**, genera predicciones sobre `val_dl`, calcula **mIoU por muestra** (sin `0/19`) y guarda **figuras** con overlay + un **log** en CSV/JSON.

**Qué hace**
- Intenta cargar `pastis_qc/best_mc.pth` (acepta tanto `{"model":...}` como `state_dict` puro).  
- Para `N_SHOW=12` muestras de validación:
  - Predice la máscara.
  - Calcula **mIoU por muestra** excluyendo `0` y `19`.
  - Reconstruye una vista RGB “bonita” con **de-normalización + stretch 2–98%**.
  - Genera una figura con **Input**, **GT (sin 0/19)** y **Pred + overlay**.
  - Guarda cada PNG en `pastis_qc/val_vis/val_<id>_miou<...>.png`.
- Exporta un **ranking** de muestras por mIoU en:
  - `pastis_qc/val_vis/val_preview_metrics.csv`
  - `pastis_qc/val_vis/val_preview_metrics.json`

**Por qué importa**
- Permite **ver** si el modelo “colorea” con sentido, no solo números.
- El CSV/JSON te deja **ordenar** y revisar casos buenos y malos rápidamente.


In [8]:
# ---- asunciones ----
BEST_CKPT   = "pastis_qc/best_mc.pth"
OUT_DIR     = Path("pastis_qc/val_vis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    N_CLASSES
except NameError:
    N_CLASSES = 20
try:
    EXCLUDE_LABELS
except NameError:
    EXCLUDE_LABELS = {0,19}
VALID_CLASSES = [c for c in range(N_CLASSES) if c not in EXCLUDE_LABELS]

# paleta simple para 0..19
PALETTE = np.array([
    [0,0,0], [230,25,75], [60,180,75], [255,225,25], [0,130,200],
    [245,130,48], [145,30,180], [70,240,240], [240,50,230], [210,245,60],
    [250,190,190], [0,128,128], [230,190,255], [170,110,40], [255,250,200],
    [128,0,0], [170,255,195], [128,128,0], [0,0,128], [128,128,128]
], dtype=np.uint8)

def colorize(mask_hw):
    m = np.clip(mask_hw, 0, len(PALETTE)-1)
    return PALETTE[m]

def denorm_and_stretch(x_chw, mean, std):
    x = x_chw.copy()
    for c in range(min(x.shape[0], len(std))):
        x[c] = x[c]*std[c] + mean[c]
    if x.shape[0] >= 3:
        vis = np.transpose(x[:3], (1,2,0))
    else:
        vis = np.repeat(x[:1].transpose(1,2,0), 3, axis=2)
    lo = np.percentile(vis, 2, axis=(0,1))
    hi = np.percentile(vis, 98, axis=(0,1))
    vis = np.clip((vis - lo) / (hi - lo + 1e-6), 0, 1)
    return vis

def sample_miou(pred_hw, gt_hw, ignore=255, valid_classes=tuple(range(1,19))):
    valid = (gt_hw != ignore)
    ious = []
    for c in valid_classes:
        p = (pred_hw == c)
        g = (gt_hw == c)
        inter = np.logical_and(p, np.logical_and(g, valid)).sum()
        union = np.logical_or(p, g)[valid].sum()
        iou_c = inter/union if union > 0 else np.nan
        ious.append(iou_c)
    miou = float(np.nanmean(ious)) if len(ious) else 0.0
    return miou, ious

# --- Carga best checkpoint  ---
if Path(BEST_CKPT).exists():
    # nada de map_object_hook; solo map_location.
    ck = torch.load(BEST_CKPT, map_location=DEVICE)  

    # Manejo flexible según cómo se guardó:
    if isinstance(ck, dict) and "model" in ck:
        model.load_state_dict(ck["model"], strict=False)
        print(f"[P5] Cargado best ckpt ep={ck.get('epoch','?')} mIoU={ck.get('miou','?')}")
    else:
        # Posible que ck sea un state_dict puro
        model.load_state_dict(ck, strict=False)
        print("[P5] Cargado best ckpt (state_dict puro).")
else:
    print("[P5] Aviso: no existe best_ckpt; se usarán pesos actuales.")


# --- Infiere y guarda N muestras de validación ---
model.eval()
N_SHOW = 12  
shown = 0
logs = []     # para guardar un CSV/JSON con mIoU por muestra

with torch.no_grad(), torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
    for batch in val_dl:
        X = batch["image"].to(DEVICE)  # (B,C,S,S)
        y = batch["mask"].cpu().numpy()
        ids = batch.get("id", [f"val_{shown+i}" for i in range(X.shape[0])])
        logits = model(X)
        pred = torch.argmax(logits, dim=1).cpu().numpy()

        for i in range(X.shape[0]):
            pid   = str(ids[i])
            xchw  = batch["image"][i].cpu().numpy()
            vis   = denorm_and_stretch(xchw, MEAN, STD)
            y_i   = y[i]
            pr_i  = pred[i]

            miou_i, _ = sample_miou(pr_i, y_i, ignore=IGNORE, valid_classes=VALID_CLASSES)

            # figuras
            fig, ax = plt.subplots(1,3, figsize=(11,3.6))
            ax[0].imshow(vis); ax[0].set_title(f"Input {pid}"); ax[0].axis("off")
            ax[1].imshow(colorize(np.where(np.isin(y_i, list(EXCLUDE_LABELS)), 0, y_i))); 
            ax[1].set_title("GT (sin 0/19)"); ax[1].axis("off")
            ax[2].imshow(vis); 
            ax[2].imshow(colorize(np.where(np.isin(pr_i, list(EXCLUDE_LABELS)), 0, pr_i)), alpha=0.45)
            ax[2].set_title(f"Pred (mIoU={miou_i:.3f})"); ax[2].axis("off")
            plt.tight_layout()

            out_png = OUT_DIR / f"val_{pid}_miou{miou_i:.3f}.png"
            fig.savefig(out_png, dpi=150, bbox_inches="tight")
            plt.close(fig)

            logs.append({"id": pid, "miou": miou_i, "file": str(out_png)})
            shown += 1
            if shown >= N_SHOW:
                break
        if shown >= N_SHOW:
            break

print(f"[P5] Guardadas {shown} visualizaciones en {OUT_DIR}")



df = pd.DataFrame(logs).sort_values("miou", ascending=False)
df.to_csv(OUT_DIR / "val_preview_metrics.csv", index=False)
with open(OUT_DIR / "val_preview_metrics.json", "w", encoding="utf-8") as f:
    json.dump(logs, f, ensure_ascii=False, indent=2)
print(f"[P5] Log: {OUT_DIR/'val_preview_metrics.csv'}")


C:\Users\LuisA\AppData\Local\Temp\ipykernel_10320\749804069.py:57: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ck = torch.load(BEST_CKPT, map_location=DEVICE)


[P5] Cargado best ckpt ep=1 mIoU=0.1263728920764196


C:\Users\LuisA\AppData\Local\Temp\ipykernel_10320\749804069.py:77: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


[P5] Guardadas 12 visualizaciones en pastis_qc\val_vis
[P5] Log: pastis_qc\val_vis\val_preview_metrics.csv
